# Recommender Systems Overview — Exercises

Companion to the note [Recommender Systems Overview](Recommender%20Systems%20Overview.md).

Practise the big picture of recommender systems: explicit vs implicit feedback, the families of approaches, the multi-stage industrial pipeline, and the core challenges (sparsity, popularity bias, feedback loops, cold start, exploration). You will build the strong non-personalised baselines from scratch on a tiny synthetic rating matrix.

**15 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from sklearn.linear_model import Ridge
rng = np.random.default_rng(7)
# Toy explicit rating matrix: 6 users x 6 items, 0 = not rated
R = np.array([[5, 4, 0, 0, 1, 0],
              [4, 0, 0, 1, 0, 0],
              [0, 5, 4, 0, 0, 0],
              [5, 0, 0, 0, 0, 2],
              [0, 4, 0, 0, 0, 0],
              [3, 0, 5, 0, 0, 0]], dtype=float)
M = R > 0                     # mask of observed entries (the set Omega)
n_users, n_items = R.shape

## Part A · Concepts

### Exercise 1 · Explicit or implicit?
*🧠 Concept · ★☆☆*

Classify each signal as **explicit** or **implicit** feedback and say what makes it noisy or sparse:

(a) a 1–5 star rating, (b) a song played for 8 seconds then skipped, (c) a product added to the cart, (d) a thumbs-down,
(e) a news article left open in a tab for 40 minutes.

Why does the note say implicit feedback has *no true negatives*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Explicit = the user deliberately states a preference. Implicit = inferred from behaviour. What does a *missing* click mean?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Explicit: (a), (d). Implicit: (b), (c), (e).

- (b) is a weak *negative-ish* signal, but a skip can also mean "not now".
- (c) is a strong positive, but carts are abandoned.
- (e) is noisy: long dwell time may just mean the tab was forgotten.

**No true negatives:** a missing interaction is ambiguous. The user may dislike the item, or may simply never have seen it.
So implicit data contains only positives plus *unlabelled* entries. That is why implicit methods use confidence weights
(WMF) or pairwise ranking losses (BPR) instead of treating zeros as dislikes ([[Collaborative Filtering and Matrix Factorization]]).

</details>

### Exercise 2 · Which family fits?
*🧠 Concept · ★☆☆*

For each scenario, pick the most suitable family from the note (popularity, content-based, CF/MF, hybrid, deep/sequential,
multimodal, graph-based, bandits) and justify in one line.

1. A brand-new shop with 50 users and no interaction history yet.
2. A news site where articles live for 24 hours.
3. A music service with 100M users and dense listening logs; next-song prediction.
4. A fashion store where taste is mostly visual and 30% of items are new every season.
5. Choosing which of 5 homepage banners to show, learning click rates online.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask for each case: is there interaction data? Do items appear faster than they collect interactions? Is order important?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Popularity** (plus onboarding questions): no data for CF yet; popularity is the strong baseline.
2. **Content-based / hybrid**: items are always cold, CF cannot learn about them before they expire.
3. **Deep sequential** (SASRec/GRU4Rec) or **implicit MF** for retrieval: lots of data and order matters.
4. **Multimodal / hybrid** ([[Multimodal Recommender Systems]]): image features score new items without interactions.
5. **Bandits** ([[RL Basics and MDPs]]): a small set of actions whose value must be learned online, explore vs exploit.

</details>

### Exercise 3 · Why multi-stage?
*🧠 Concept · ★★☆*

Explain why industrial systems use **retrieval → ranking → re-ranking** instead of one model that scores every item.
What is each stage optimised for (recall, precision, business goals)? Which stage would you put a diversity rule in, and why not earlier?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about how many items each stage sees and how expensive each model is per item.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Retrieval** sees millions of items, so it must be cheap: two-tower embeddings + approximate nearest neighbours. It is tuned for **recall**: do not lose good items.
- **Ranking** sees only hundreds to thousands of candidates, so it can afford a rich model (GBDT, DLRM, Wide & Deep) with many cross features. It is tuned for **precision** at the top (CTR, watch time).
- **Re-ranking** works on the final short list and applies **diversity, freshness, business rules and fairness** constraints.

Diversity is a property of the *list*, not of a single item. You can only judge it once you know which items made the
short list, so it belongs in re-ranking. Applying it in retrieval would throw away relevant candidates too early.

</details>

### Exercise 4 · Feedback loops and popularity bias
*🧠 Concept · ★★☆*

A system trains every night on yesterday's clicks, and users can only click what was recommended.
Describe the **feedback loop** this creates. Why does it amplify **popularity bias**, and why does offline evaluation on
logged data look *better* and better while real quality may degrade? Name two mitigations.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Clicks are produced *by* the recommender's exposure decisions, then used to train the next recommender.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Exposure → clicks → training data → exposure. Items that are recommended get clicks, so the model learns they are good and
recommends them more. Items never shown never get clicks and are never learned. This is a self-fulfilling prophecy (a rich-get-richer dynamic).

Offline metrics computed on logged data reward reproducing the old policy, since the logged positives are exactly the
items it chose to show. So the metric goes up while catalogue coverage and user satisfaction may fall.

Mitigations: **exploration** (ε-greedy, bandits, random slots), **inverse propensity weighting** of logged clicks
([[Evaluating Recommenders]]), diversity/novelty constraints in re-ranking, and popularity-debiased losses or negative sampling.

</details>

## Part B · By hand

### Exercise 5 · Sparsity of the toy matrix
*✍️ By hand · ★☆☆*

Look at the matrix `R` in the setup cell (6 users × 6 items, 0 = missing). Compute its **density**
$|\Omega| / (|U|\cdot|I|)$. Then compute the density of a real-world matrix with 1M users, 100k items and 50M interactions.

In [ ]:
density_toy = None
density_real = None

check('toy density', density_toy, M.mean())
check('real density', density_real, 50e6 / (1e6 * 1e5))

<details>
<summary><b>💡 Hint</b></summary>

Count non-zeros row by row: 3, 2, 2, 2, 1, 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$|\Omega| = 12$, so density $= 12/36 = 1/3 \approx 0.333$.
Real: $5\cdot 10^7 / 10^{11} = 5\cdot 10^{-4} = 0.05\%$. Real matrices are 99.95% empty, which is why we never store them densely
and why similarity estimates between users are noisy (few co-rated items).

```python
density_toy = 12 / 36
density_real = 5e-4
```

</details>

### Exercise 6 · Mean rating vs damped mean
*✍️ By hand · ★★☆*

Ranking items by their raw mean rating favours items with very few ratings. A common fix is the **damped (Bayesian) mean**
$$\tilde r_i = \frac{\sum_{u\in\Omega_i} r_{ui} + m\,\mu}{n_i + m},$$
where $\mu$ is the global mean rating and $m$ a pseudo-count.

For items 0, 1, 2 of `R`, compute the raw mean and the damped mean with $m=3$. Which item is #1 under each rule?
Store the damped means of items 0, 1, 2 in `damped_012` and the index of the best item under the damped mean in `best_damped`.

In [ ]:
damped_012 = None   # [item0, item1, item2]
best_damped = None

_mu = R[M].mean(); _n = M.sum(0); _s = R.sum(0)
_ref = (_s + 3 * _mu) / (_n + 3)
check('damped means', damped_012, _ref[:3], tol=1e-3)
check('best item', best_damped, int(np.argmax(_ref)))

<details>
<summary><b>💡 Hint 1</b></summary>

Global mean: sum of all 12 ratings is 43, so $\mu = 43/12 \approx 3.583$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Item 0: ratings 5,4,5,3 (sum 17, n=4). Item 1: 4,5,4 (sum 13, n=3). Item 2: 4,5 (sum 9, n=2).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Raw means: item 0 $=4.25$, item 1 $\approx 4.333$, item 2 $=4.5$, so **item 2** wins with only 2 ratings.

With $m=3$, $m\mu = 10.75$:
- item 0: $(17+10.75)/7 \approx 3.964$
- item 1: $(13+10.75)/6 \approx 3.958$
- item 2: $(9+10.75)/5 = 3.95$

Now **item 0** wins: its 4 ratings make its mean more trustworthy. The prior pulls every mean towards $\mu$, and pulls it harder when $n_i$ is small.
(This is the posterior mean under a conjugate prior, see [[Bayesian Inference]].)

```python
damped_012 = [27.75 / 7, 23.75 / 6, 19.75 / 5]
best_damped = 0
```

</details>

### Exercise 7 · From play counts to preference and confidence
*✍️ By hand · ★☆☆*

In implicit weighted MF (Hu, Koren & Volinsky) the raw count $r_{ui}$ becomes a binary **preference** $p_{ui}=\mathbb 1[r_{ui}>0]$
and a **confidence** $c_{ui} = 1 + \alpha r_{ui}$. For play counts $r = (0, 1, 3, 10)$ and $\alpha = 40$, compute $p$ and $c$.
Why is the confidence of the zero entry not 0?

In [ ]:
pref = None   # list of 4
conf = None   # list of 4

_r = np.array([0, 1, 3, 10.])
check('preference', pref, (_r > 0).astype(float))
check('confidence', conf, 1 + 40 * _r)

<details>
<summary><b>💡 Hint</b></summary>

Plug in directly. For the zero entry: does the model ignore it, or treat it as a weak negative?

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p = (0, 1, 1, 1)$, $c = (1, 41, 121, 401)$.
The unobserved entry still gets confidence 1: it is a **weak negative**, used in the loss but with much less weight than
observed plays. Setting it to 0 would remove all negatives and the trivial solution "predict 1 everywhere" would be optimal.

```python
pref = [0, 1, 1, 1]
conf = [1, 41, 121, 401]
```

</details>

### Exercise 8 · The long tail
*✍️ By hand · ★☆☆*

A catalogue of 10 items received these interaction counts: $(50, 20, 10, 8, 5, 3, 2, 1, 1, 0)$.
What share of all interactions goes to the top 20% of items? What share of items has at most 1 interaction?

In [ ]:
share_top20 = None
share_tail = None

_c = np.array([50, 20, 10, 8, 5, 3, 2, 1, 1, 0])
check('top-20% share', share_top20, np.sort(_c)[::-1][:2].sum() / _c.sum())
check('tail share', share_tail, np.mean(_c <= 1))

<details>
<summary><b>💡 Hint</b></summary>

Total is 100. Top 20% of 10 items = 2 items.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Top 2 items: $(50+20)/100 = 0.70$. Items with $\le 1$ interaction: 3 of 10 $= 0.30$.
This head/tail shape is typical. A model trained on such data sees tail items rarely, which is one source of popularity bias.

```python
share_top20 = 0.70
share_tail = 0.30
```

</details>

### Exercise 9 · Latency budget of a multi-stage pipeline
*✍️ By hand · ★☆☆*

A ranking model needs 0.05 ms per item. The catalogue has $10^7$ items.

(a) How long (seconds) would scoring every item for one request take?
(b) With ANN retrieval (5 ms) returning 500 candidates which are then ranked, what is the total latency in seconds?

In [ ]:
t_full = None    # seconds
t_staged = None  # seconds

check('full scoring', t_full, 1e7 * 0.05e-3)
check('staged', t_staged, 5e-3 + 500 * 0.05e-3)

<details>
<summary><b>💡 Hint</b></summary>

0.05 ms = $5\cdot 10^{-5}$ s.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $10^7 \cdot 5\cdot10^{-5} = 500$ s, which is hopeless for a web request.
(b) $5\,\text{ms} + 500\cdot 0.05\,\text{ms} = 5 + 25 = 30$ ms $= 0.03$ s.
The cheap retrieval stage makes the expensive ranker affordable.

```python
t_full = 500.0
t_staged = 0.03
```

</details>

## Part C · Baselines in code

### Exercise 10 · Most-popular recommender
*💻 Code · ★☆☆*

Implement `recommend_popular(R, u, K)` returning the indices of the `K` items with the **most ratings** (count of non-zeros)
that user `u` has **not** rated yet, most popular first. Break ties by the lower item index.

In [ ]:
def recommend_popular(R, u, K):
    # TODO: popularity = number of observed ratings per item; exclude items u already rated
    return None

recs_u4 = recommend_popular(R, 4, 3)

_cnt = (R > 0).sum(0)
_ref = sorted([i for i in range(R.shape[1]) if R[4, i] == 0], key=lambda i: (-_cnt[i], i))[:3]
check('top-3 for user 4', recs_u4, _ref)

<details>
<summary><b>💡 Hint</b></summary>

`np.argsort(-pop, kind='stable')` keeps lower indices first for ties. Filter out items with `R[u] > 0`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

User 4 has rated only item 1. Counts are $(4,3,2,1,1,1)$, so the recommendations are items 0, 2, 3.
It is non-personalised, yet in many benchmarks it is surprisingly hard to beat by a wide margin.

```python
def recommend_popular(R, u, K):
    pop = (R > 0).sum(0)
    order = np.argsort(-pop, kind='stable')
    return [int(i) for i in order if R[u, i] == 0][:K]

recs_u4 = recommend_popular(R, 4, 3)
print(recs_u4)
```

</details>

### Exercise 11 · Catalogue coverage
*💻 Code · ★☆☆*

**Catalogue coverage** is the fraction of items that appear in at least one user's top-K list.
Implement `coverage(rec_lists, n_items)` and compute the coverage of the popularity recommender with K=2 over all 6 users.
Why is coverage a useful sanity metric for popularity-style models?

In [ ]:
def coverage(rec_lists, n_items):
    return None

cov_pop = None

if cov_pop is not None:
    _lists = [recommend_popular(R, u, 2) for u in range(n_users)]
    check('coverage', cov_pop, len(set(np.concatenate(_lists).tolist())) / n_items)
else:
    check('coverage', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

Union of all lists, as a Python `set`, divided by the number of items.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The lists only contain the few most popular items each user has not yet seen, so coverage is limited. Here it is $4/6 \approx 0.67$, and items 4 and 5 are never shown. On a real catalogue with
millions of items, a popularity list covers far below 1%. Accuracy metrics alone can hide that a model only ever shows the head.

```python
def coverage(rec_lists, n_items):
    return len(set(int(i) for lst in rec_lists for i in lst)) / n_items

cov_pop = coverage([recommend_popular(R, u, 2) for u in range(n_users)], n_items)
print(cov_pop)
```

</details>

### Exercise 12 · Bias baseline: μ + b_u + b_i
*💻 Code · ★★☆*

The classic baseline predictor is $\hat r_{ui} = \mu + b_u + b_i$, fitted on observed entries by
$$\min_{\mu,b}\sum_{(u,i)\in\Omega}(r_{ui}-\mu-b_u-b_i)^2 + \lambda\Big(\sum_u b_u^2+\sum_i b_i^2\Big).$$
This is ridge regression on a one-hot design matrix (one column per user, one per item) with an unpenalised intercept $\mu$.

Implement `fit_bias_baseline(R, lam)` returning `(mu, b_user, b_item)` and use it with $\lambda=1$.
Compare with `sklearn.linear_model.Ridge`.

In [ ]:
def fit_bias_baseline(R, lam):
    # TODO: build one-hot design for observed (u, i), centre, solve ridge, recover mu
    return None

bias_fit = fit_bias_baseline(R, 1.0)

_u, _i = np.nonzero(R)
_X = np.zeros((len(_u), n_users + n_items)); _X[np.arange(len(_u)), _u] = 1; _X[np.arange(len(_u)), n_users + _i] = 1
_rd = Ridge(alpha=1.0).fit(_X, R[_u, _i])
if bias_fit is not None:
    check('mu', bias_fit[0], _rd.intercept_)
    check('user + item biases', np.r_[bias_fit[1], bias_fit[2]], _rd.coef_)
else:
    check('bias baseline', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Rows of the design matrix: one per observed $(u,i)$, with a 1 in column $u$ and a 1 in column $n_{users}+i$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

As in [[Linear Regression]] ridge: centre $X$ and $y$, solve $(X_c^\top X_c + \lambda I)w = X_c^\top y_c$, then $\mu = \bar y - \bar x^\top w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Centring removes the intercept so only the biases are penalised. Note that the fitted $\mu$ is *not* exactly the global mean,
because the biases are shrunk and do not average out to zero. The baseline already captures "harsh users" and "great items".
It is the $\mu + b_u + b_i$ part of the MF model in [[Collaborative Filtering and Matrix Factorization]].

```python
def fit_bias_baseline(R, lam):
    u, i = np.nonzero(R)
    nu, ni = R.shape
    X = np.zeros((len(u), nu + ni))
    X[np.arange(len(u)), u] = 1
    X[np.arange(len(u)), nu + i] = 1
    y = R[u, i]
    xm, ym = X.mean(0), y.mean()
    Xc, yc = X - xm, y - ym
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(nu + ni), Xc.T @ yc)
    return ym - xm @ w, w[:nu], w[nu:]

bias_fit = fit_bias_baseline(R, 1.0)
print('mu =', round(bias_fit[0], 3), 'b_item =', np.round(bias_fit[2], 2))
```

</details>

### Exercise 13 · Gini coefficient of popularity
*💻 Code · ★★☆*

The **Gini coefficient** measures how concentrated interactions are (0 = all items equally popular, →1 = one item gets everything):
$$G = \frac{\sum_i\sum_j |x_i - x_j|}{2 n^2 \bar x}.$$
Implement it in $O(n\log n)$ with the sorted formula $G = \frac{2\sum_{k=1}^n k\,x_{(k)}}{n\sum_k x_k} - \frac{n+1}{n}$
(where $x_{(1)}\le\dots\le x_{(n)}$) and apply it to the long-tail counts of Exercise 8.

In [ ]:
def gini(x):
    # TODO: sorted O(n log n) formula
    return None

tail_counts = np.array([50, 20, 10, 8, 5, 3, 2, 1, 1, 0])
g_tail = gini(tail_counts)

_x = tail_counts.astype(float)
check('Gini', g_tail, np.abs(_x[:, None] - _x[None, :]).sum() / (2 * len(_x) ** 2 * _x.mean()))

<details>
<summary><b>💡 Hint</b></summary>

`x = np.sort(x)`; `k = np.arange(1, n + 1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$G \approx 0.648$ for these counts: strongly concentrated. Use it to compare how concentrated *recommendations* are vs. the
data (does the model amplify popularity?).

```python
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    k = np.arange(1, n + 1)
    return 2 * np.sum(k * x) / (n * x.sum()) - (n + 1) / n

tail_counts = np.array([50, 20, 10, 8, 5, 3, 2, 1, 1, 0])
g_tail = gini(tail_counts)
print(round(g_tail, 3))
```

</details>

### Exercise 14 · Simulating a feedback loop
*💻 Code · ★★★*

Simulate 50 items with true appeal $a_i\sim U(0.05, 0.3)$ and 100 users over 30 rounds. In each round every user is shown 5 items
and clicks each shown item independently with probability $a_i$.

- **Random exposure:** the 5 items are chosen uniformly at random.
- **Popularity loop:** the 5 items are the current top-5 by accumulated clicks (ties by index).

Write `simulate(loop, seed)` that returns the final click counts. Compute the Gini of both (`g_random`, `g_loop`) using your `gini`.
How many distinct items ever get a click under the loop?

In [ ]:
def simulate(loop, seed=1, rounds=30, U=100, I=50):
    # TODO
    return None

g_random = None
g_loop = None

if g_loop is not None:
    check('loop concentrates clicks far more than random exposure', g_loop > g_random + 0.3, True)
    _c = simulate(True)
    _x = np.asarray(_c, float)
    check('your gini on the loop clicks', g_loop, np.abs(_x[:, None] - _x[None, :]).sum() / (2 * len(_x) ** 2 * _x.mean()))
else:
    check('feedback loop', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Keep a `clicks = np.zeros(I)` vector; for the loop use `np.argsort(-clicks, kind='stable')[:5]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Clicks for the shown items: `shown[rng.random(5) < appeal[shown]]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With random exposure the Gini is about 0.24, roughly what the appeal spread explains. Under the popularity loop the very first
top-5 (initially just items 0–4, since all counts are 0!) are the only ones ever shown: $G\approx 0.92$ and only 5 items ever
get a click, regardless of their true appeal. The data "confirms" the policy. This is why systems need exploration.

```python
def simulate(loop, seed=1, rounds=30, U=100, I=50):
    rng_s = np.random.default_rng(seed)
    appeal = rng_s.uniform(0.05, 0.3, I)
    clicks = np.zeros(I)
    for _ in range(rounds):
        for _u in range(U):
            if loop:
                shown = np.argsort(-clicks, kind='stable')[:5]
            else:
                shown = rng_s.choice(I, 5, replace=False)
            clicks[shown[rng_s.random(5) < appeal[shown]]] += 1
    return clicks

g_random = gini(simulate(False))
g_loop = gini(simulate(True))
print(round(g_random, 3), round(g_loop, 3), 'items ever clicked:', int((simulate(True) > 0).sum()))
```

</details>

### Exercise 15 · ε-greedy exploration
*💻 Code · ★★★*

Five banners have unknown click-through rates `ctr = [0.02, 0.05, 0.04, 0.03, 0.06]`. Over $T=5000$ impressions, a
**greedy** policy always shows the banner with the highest estimated CTR so far; **ε-greedy** does the same but with probability
ε shows a uniformly random banner.

Implement `run_bandit(eps, T, seed)` returning `(total_clicks, pulls_per_arm)`. Compare ε=0 with ε=0.1 (seed 0).
Store the fraction of ε-greedy pulls spent on the best arm in `best_share`.

In [ ]:
ctr = np.array([0.02, 0.05, 0.04, 0.03, 0.06])

def run_bandit(eps, T=5000, seed=0):
    # TODO: track pulls n[a] and clicks s[a]; estimate = s / max(n, 1)
    return None

best_share = None

if best_share is not None:
    _g = run_bandit(0.0); _e = run_bandit(0.1)
    check('eps-greedy beats greedy', _e[0] > _g[0], True)
    check('eps-greedy mostly plays the best arm', best_share > 0.5 and np.argmax(_e[1]) == np.argmax(ctr), True)
else:
    check('bandit', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Greedy with all estimates at 0 picks arm 0 (argmax tie). If arm 0 never clicks early, does its estimate ever lose the tie?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Greedy gets stuck: all estimates start at 0, `argmax` picks arm 0, and since the others are never tried their estimates stay 0.
It plays arm 0 all 5000 times (≈95 clicks). ε-greedy keeps sampling all arms, discovers arm 4 and plays it ≈59% of the time
(≈246 clicks). This explore/exploit trade-off is the bandit view of recommendation (see [[RL Basics and MDPs]]).

```python
ctr = np.array([0.02, 0.05, 0.04, 0.03, 0.06])

def run_bandit(eps, T=5000, seed=0):
    rng_b = np.random.default_rng(seed)
    n = np.zeros(len(ctr)); s = np.zeros(len(ctr)); total = 0
    for _ in range(T):
        if rng_b.random() < eps:
            a = rng_b.integers(len(ctr))
        else:
            a = int(np.argmax(s / np.maximum(n, 1)))
        click = rng_b.random() < ctr[a]
        n[a] += 1; s[a] += click; total += click
    return int(total), n

tot_e, pulls_e = run_bandit(0.1)
best_share = pulls_e[np.argmax(ctr)] / pulls_e.sum()
print('greedy:', run_bandit(0.0)[0], 'eps-greedy:', tot_e, 'best share:', round(best_share, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Recommender Systems Overview](Recommender%20Systems%20Overview.md).*